In [44]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

In [45]:
df=pd.read_csv('diabetes.csv')

In [46]:
df.sample(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
115,4,146,92.0,32,169.5,31.2,0.539,61,1
422,0,102,64.0,46,78.0,40.6,0.496,21,0
352,3,61,82.0,28,102.5,34.4,0.243,46,0
309,2,124,68.0,28,205.0,32.9,0.875,30,1
472,0,119,66.0,27,102.5,38.8,0.259,22,0
530,2,122,60.0,18,106.0,29.8,0.717,22,0
703,2,129,70.0,27,102.5,38.5,0.304,41,0
22,7,196,90.0,32,169.5,39.8,0.451,41,1
492,4,99,68.0,38,102.5,32.8,0.145,33,0
260,3,191,68.0,15,130.0,30.9,0.299,34,0


In [47]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.495990
BloodPressure               0.174469
SkinThickness               0.295138
Insulin                     0.377081
BMI                         0.315577
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [48]:
x=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [49]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [50]:
X=scaler.fit_transform(x)


In [51]:
X

array([[ 0.63994726,  0.86462486, -0.03218035, ...,  0.16948251,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.20472661, -0.52812374, ..., -0.84854874,
        -0.36506078, -0.19067191],
       [ 1.23388019,  2.01426457, -0.69343821, ..., -1.32847775,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 , -0.02224005, -0.03218035, ..., -0.90672195,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.14199419, -1.02406713, ..., -0.33953311,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.94195182, -0.19749482, ..., -0.2959032 ,
        -0.47378505, -0.87137393]], shape=(768, 8))

In [52]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [53]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [54]:
model=Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [55]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4153 - loss: 0.7456 - val_accuracy: 0.5779 - val_loss: 0.7016
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6368 - loss: 0.6590 - val_accuracy: 0.6494 - val_loss: 0.6292
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6922 - loss: 0.5966 - val_accuracy: 0.6753 - val_loss: 0.5756
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7068 - loss: 0.5496 - val_accuracy: 0.7078 - val_loss: 0.5370
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7345 - loss: 0.5150 - val_accuracy: 0.7532 - val_loss: 0.5082
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7590 - loss: 0.4880 - val_accuracy: 0.7662 - val_loss: 0.4871
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7752 - loss: 0.4698 - val_accuracy: 0.7727 - val_loss: 0.4732
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7818 - loss: 0.4558 - val_accuracy: 0.7922 - 

In [56]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (614, 8)
y_train: (614,)
X_test: (154, 8)
y_test: (154,)


1. How to select appropriate optimizer.
2. No. of node in the layer.
3. How to select no. of layers.
4. All in one model

In [57]:
# pip install Keras_tuner 

In [58]:
import kerastuner as kt

In [59]:
def build_model(hp): 
    model=Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))
    optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
    model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [60]:
tuner=kt.RandomSearch(build_model,
                     objective='val_accuracy',
                     max_trials=5)

Reloading Tuner from .\untitled_project\tuner0.json


In [61]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [62]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [63]:
model=tuner.get_best_models(num_models=1)[0]

In [65]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [66]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7622 - loss: 0.4909 - val_accuracy: 0.7143 - val_loss: 0.5126
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7736 - loss: 0.4693 - val_accuracy: 0.7468 - val_loss: 0.4957
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7704 - loss: 0.4554 - val_accuracy: 0.7597 - val_loss: 0.4857
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7736 - loss: 0.4461 - val_accuracy: 0.7532 - val_loss: 0.4786
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7720 - loss: 0.4380 - val_accuracy: 0.7597 - val_loss: 0.4712
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7801 - loss: 0.4317 - val_accuracy: 0.7468 - val_loss: 0.4671
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7850 - loss: 0.4259 - val_accuracy: 0.7597 - val_loss: 0.4636
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7899 - loss: 0.4214 - val_accuracy: 0.76

In [72]:
def build_model(hp): 
    model=Sequential()
    units=hp.Int('units',min_value=8,max_value=128,step=8)
    model.add(Dense(units=units,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
    return model 

In [75]:
tuner=kt.RandomSearch(build_model,
                     objective='val_accuracy', 
                     max_trials=5, 
                     directory='mydir', 
                     project_name='suraj')

In [76]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7207792401313782

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 15s


In [77]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [78]:
model=tuner.get_best_models(num_models=1)[0]

In [81]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test, y_test)
)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9055 - loss: 0.2462 - val_accuracy: 0.8377 - val_loss: 0.4804
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9007 - loss: 0.2445 - val_accuracy: 0.8377 - val_loss: 0.4820
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9023 - loss: 0.2444 - val_accuracy: 0.8442 - val_loss: 0.4799
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9007 - loss: 0.2453 - val_accuracy: 0.8377 - val_loss: 0.4826
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9088 - loss: 0.2426 - val_accuracy: 0.8377 - val_loss: 0.4857
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9039 - loss: 0.2425 - val_accuracy: 0.8377 - val_loss: 0.4853
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9072 - loss: 0.2427 - val_accuracy: 0.8571 - val_loss: 0.4834
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9007 - loss: 0.2424 - val_accuracy: 0.84

In [82]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [83]:
model=tuner.get_best_models(num_models=1)[0]

In [84]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7818 - loss: 0.4709 - val_accuracy: 0.7922 - val_loss: 0.4558
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7948 - loss: 0.4388 - val_accuracy: 0.7987 - val_loss: 0.4413
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7980 - loss: 0.4241 - val_accuracy: 0.7987 - val_loss: 0.4349
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8029 - loss: 0.4141 - val_accuracy: 0.7987 - val_loss: 0.4338
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8029 - loss: 0.4056 - val_accuracy: 0.8052 - val_loss: 0.4326
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8078 - loss: 0.3983 - val_accuracy: 0.7987 - val_loss: 0.4296
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8127 - loss: 0.3922 - val_accuracy: 0.8117 - val_loss: 0.4271
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8208 - loss: 0.3852 - val_accuracy: 0.8182 - 

In [85]:
# How to select no. of layers

In [90]:
def build_model(hp): 
    model=Sequential()
    model.add(Dense(72,activation='relu',input_dim=8))
    for i in range(hp.Int)('num_layers',min_value=1,max_value=10): 
        model.add(Densen(72,activation='relu'))
        model.add(Dense(1,activation='sigmoid'))
        model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
        return model

In [110]:
tuner=kt.RandomSearch(build_model, 
                     objective='val_accuracy', 
                     max_trials=3,
                      directory='mydir',
                     project_name='num_layers')

TypeError: 'method' object cannot be interpreted as an integer

In [104]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [105]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [106]:
model=tuner.get_best_models(num_models=1)[0]

TypeError: 'method' object cannot be interpreted as an integer

In [102]:
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9039 - loss: 0.2421 - val_accuracy: 0.8442 - val_loss: 0.4924
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9072 - loss: 0.2418 - val_accuracy: 0.8506 - val_loss: 0.4917
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9088 - loss: 0.2400 - val_accuracy: 0.8377 - val_loss: 0.4917
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9039 - loss: 0.2416 - val_accuracy: 0.8312 - val_loss: 0.5007
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9055 - loss: 0.2397 - val_accuracy: 0.8312 - val_loss: 0.4982
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9072 - loss: 0.2388 - val_accuracy: 0.8442 - val_loss: 0.4951
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9088 - loss: 0.2385 - val_accuracy: 0.8442 - val_loss: 0.4948
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9055 - loss: 0.2390 - val_accuracy: 0.844